# Submit the Titanic Spark Training Pipeline

Submit [`titanic-spark-training-pipeline.yml`](./titanic-spark-training-pipeline.yml), stream its three Spark 3.5 jobs, download the trained Spark ML model and metrics from ADLS Gen2, and register the validated model in Azure Machine Learning.

In [1]:
from pathlib import Path, PurePosixPath
import json
import os
import shutil

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model
from azure.identity import AzureCliCredential
from azure.storage.filedatalake import DataLakeServiceClient
from dotenv import load_dotenv


def find_workshop_root() -> Path:
    notebook_file = globals().get("__vsc_ipynb_file__")
    start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AGENDA.md").is_file() and (candidate / ".env.example").is_file():
            return candidate
        nested = candidate / "workshop"
        if (nested / "AGENDA.md").is_file() and (nested / ".env.example").is_file():
            return nested
    raise FileNotFoundError("Run this notebook from inside the workshop folder")


def require_env(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set in workshop/.env")
    return value


def env_flag(name: str) -> bool:
    value = os.getenv(name, "false").strip().lower()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be exactly true or false")
    return value == "true"


def download_adls_directory(
    service_client: DataLakeServiceClient,
    filesystem: str,
    prefix: str,
    destination: Path,
) -> None:
    file_system_client = service_client.get_file_system_client(filesystem)
    normalized_prefix = prefix.strip("/")
    destination.mkdir(parents=True, exist_ok=True)
    found_file = False
    for remote_path in file_system_client.get_paths(
        path=normalized_prefix,
        recursive=True,
    ):
        relative = PurePosixPath(remote_path.name).relative_to(normalized_prefix)
        local_path = destination.joinpath(*relative.parts)
        if remote_path.is_directory:
            local_path.mkdir(parents=True, exist_ok=True)
            continue
        found_file = True
        local_path.parent.mkdir(parents=True, exist_ok=True)
        file_client = file_system_client.get_file_client(remote_path.name)
        local_path.write_bytes(file_client.download_file().readall())
    if not found_file:
        raise FileNotFoundError(
            f"No files found under abfss://{filesystem}/{normalized_prefix}"
        )


WORKSHOP_ROOT = find_workshop_root()
PIPELINE_PATH = WORKSHOP_ROOT / "notebooks/06_spark_jobs_pipelines/titanic-spark-training-pipeline.yml"
OUTPUT_ROOT = WORKSHOP_ROOT / "outputs/titanic_spark_training"
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

SUBSCRIPTION_ID = require_env("AZURE_SUBSCRIPTION_ID")
RESOURCE_GROUP = require_env("AZURE_RESOURCE_GROUP")
WORKSPACE_NAME = require_env("AZUREML_WORKSPACE_NAME")
MODEL_NAME = require_env("TITANIC_SPARK_MODEL_NAME")
STORAGE_ACCOUNT = require_env("TITANIC_SPARK_STORAGE_ACCOUNT")
FILESYSTEM = require_env("TITANIC_SPARK_FILESYSTEM")
MODEL_PREFIX = require_env("TITANIC_SPARK_MODEL_PATH")
METRICS_PREFIX = require_env("TITANIC_SPARK_METRICS_PATH")
VALIDATION_PREFIX = require_env("TITANIC_SPARK_VALIDATION_PATH")
RUN_PIPELINE = env_flag("RUN_TITANIC_SPARK_PIPELINE")
REGISTER_MODEL = env_flag("REGISTER_TITANIC_SPARK_MODEL")

credential = AzureCliCredential(
    tenant_id=os.getenv("AZURE_TENANT_ID") or None,
    process_timeout=60,
)
ml_client = MLClient(
    credential,
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE_NAME,
)
data_lake_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential,
)

print(f"Workspace: {WORKSPACE_NAME}")
print(f"Pipeline YAML: {PIPELINE_PATH}")
print(f"Model name: {MODEL_NAME}")
print(f"Run enabled: {RUN_PIPELINE}; registration enabled: {REGISTER_MODEL}")

Workspace: mlwpublicncus04
Pipeline YAML: C:\Users\{user}\Documents\MLOPs-AzureML\workshop\notebooks\06_spark_jobs_pipelines\titanic-spark-training-pipeline.yml
Model name: workshop-titanic-spark-model
Run enabled: True; registration enabled: True


In [2]:
pipeline_job = load_job(PIPELINE_PATH)
spark_jobs = list(pipeline_job.jobs.items())
expected_jobs = ["prepare_data", "train_model", "score_test_data", "validate_model"]
if [name for name, _ in spark_jobs] != expected_jobs:
    raise ValueError(f"Pipeline jobs must remain {expected_jobs}")
for name, spark_job in spark_jobs:
    if spark_job.type != "spark":
        raise TypeError(f"{name} must be a Spark job")
    if str(spark_job.resources.runtime_version) != "3.5":
        raise ValueError(f"{name} must use Spark runtime 3.5")
    print(
        f"{name}: runtime={spark_job.resources.runtime_version}; "
        f"instance={spark_job.resources.instance_type}"
    )

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


prepare_data: runtime=3.5; instance=standard_e4s_v3
train_model: runtime=3.5; instance=standard_e4s_v3
evaluate_model: runtime=3.5; instance=standard_e4s_v3
validate_model: runtime=3.5; instance=standard_e4s_v3


In [ ]:
if not RUN_PIPELINE:
    print("Submission disabled. Set RUN_TITANIC_SPARK_PIPELINE=true in workshop/.env.")
else:
    submitted = ml_client.jobs.create_or_update(pipeline_job)
    print(f"Submitted pipeline: {submitted.name}")
    print(f"Studio: {submitted.studio_url}")
    ml_client.jobs.stream(submitted.name)
    final_job = ml_client.jobs.get(submitted.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Pipeline ended with status {final_job.status}")

    run_output = OUTPUT_ROOT / final_job.name
    if run_output.exists():
        shutil.rmtree(run_output)
    model_download = run_output / "pipeline_model"
    metrics_download = run_output / "pipeline_metrics"
    validation_download = run_output / "pipeline_validation"
    download_adls_directory(
        data_lake_client,
        FILESYSTEM,
        MODEL_PREFIX,
        model_download,
    )
    download_adls_directory(
        data_lake_client,
        FILESYSTEM,
        METRICS_PREFIX,
        metrics_download,
    )
    download_adls_directory(
        data_lake_client,
        FILESYSTEM,
        VALIDATION_PREFIX,
        validation_download,
    )

    spark_model_path = model_download / "sparkml-model"
    if not (spark_model_path / "metadata").is_dir():
        raise FileNotFoundError(f"Spark ML model metadata missing from {spark_model_path}")
    metric_files = sorted((metrics_download / "metrics").glob("part-*.json"))
    if len(metric_files) != 1:
        raise FileNotFoundError("Expected exactly one metrics JSON part file")
    metrics = json.loads(metric_files[0].read_text(encoding="utf-8").splitlines()[0])
    if float(metrics["accuracy"]) < 0.70:
        raise RuntimeError(f"Accuracy below threshold: {metrics['accuracy']}")
    print(json.dumps(metrics, indent=2, sort_keys=True))

    validation_files = sorted(validation_download.glob("part-*.json"))
    if len(validation_files) != 1:
        raise FileNotFoundError("Expected exactly one validation JSON part file")
    validation = json.loads(
        validation_files[0].read_text(encoding="utf-8").splitlines()[0]
    )
    if validation.get("validated") is not True:
        raise RuntimeError(f"Model validation did not pass: {validation}")
    print(json.dumps(validation, indent=2, sort_keys=True))

    if REGISTER_MODEL:
        model_definition = Model(
            name=MODEL_NAME,
            path=str(model_download),
            type=AssetTypes.CUSTOM_MODEL,
            description="Spark ML Random Forest trained by the Titanic Spark 3.5 pipeline",
            tags={
                "workshop": "azureml-h2o",
                "framework": "sparkml",
                "spark_runtime": "3.5",
                "pipeline_job": final_job.name,
                "accuracy": str(metrics["accuracy"]),
            },
        )
        registered_model = ml_client.models.create_or_update(model_definition)
        verified_model = ml_client.models.get(
            MODEL_NAME,
            version=registered_model.version,
        )
        print(f"Registered model: {verified_model.name}:{verified_model.version}")
    else:
        print("Model registration disabled.")

    print(f"Final pipeline status: {final_job.status}")
    print(f"Downloaded model: {model_download}")
    print(f"Downloaded metrics: {metrics_download}")
    print(f"Downloaded validation: {validation_download}")

Uploading pipeline (0.01 MBs): 100%|##########| 10375/10375 [00:00<00:00, 19569.35it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Submitted pipeline: musing_date_ws9pfpkm52
Studio: https://ml.azure.com/runs/musing_date_ws9pfpkm52?wsid=/subscriptions/{subscription-id}/resourcegroups/rg-aml-ws-public-ncus-04/workspaces/mlwpublicncus04&tid={tenant-id}
RunId: musing_date_ws9pfpkm52
Web View: https://ml.azure.com/runs/musing_date_ws9pfpkm52?wsid=/subscriptions/{subscription-id}/resourcegroups/rg-aml-ws-public-ncus-04/workspaces/mlwpublicncus04

Streaming logs/azureml/executionlogs.txt

[2026-10-06 14:13:56Z] Submitting 1 runs, first five are: 00ab8862:e70845c1-8f76-4fe7-b396-9784ef851cb8
[2026-10-06 14:16:57Z] Completing processing run id e70845c1-8f76-4fe7-b396-9784ef851cb8.
[2026-10-06 14:17:01Z] Submitting 1 runs, first five are: 7e8a4509:0f714514-6811-4974-a0c5-8d84dcd1685f
[2026-10-06 14:19:41Z] Completing processing run id 0f714514-6811-4974-a0c5-8d84dcd1685f.
[2026-10-06 14:19:48Z] Submitting 1 runs, first five are: 1ce74c4d:b15bd896-eb6f-4602-9701-fd6ac9446ab1
[2026-10-06 14:22:41Z] Completing processing run i

Uploading pipeline_model (0.16 MBs): 100%|##########| 157942/157942 [00:00<00:00, 168749.47it/s]




Registered model: workshop-titanic-spark-model:1
Final pipeline status: Completed
Downloaded model: C:\Users\{user}\Documents\MLOPs-AzureML\workshop\outputs\titanic_spark_training\musing_date_ws9pfpkm52\pipeline_model
Downloaded metrics: C:\Users\{user}\Documents\MLOPs-AzureML\workshop\outputs\titanic_spark_training\musing_date_ws9pfpkm52\pipeline_metrics
Downloaded validation: C:\Users\{user}\Documents\MLOPs-AzureML\workshop\outputs\titanic_spark_training\musing_date_ws9pfpkm52\pipeline_validation


: 

## Expected result

All four Spark 3.5 stages complete, evaluation accuracy is at least 0.70, `validate_model` emits `validated=true`, the artifacts are downloaded under `workshop/outputs/titanic_spark_training/<run-name>/`, and only then is the custom model registered.